# **CIROH Datasets in HydroShare** - Download and chunk dataset information for CIROH AI Bot

In [2]:
import os
import requests
from dotenv import load_dotenv
from hsclient import HydroShare
import rdflib
import json
from typing import Dict, Any, List, Optional, Tuple
import re
import html
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
from database import DatabaseManager

db_manager = DatabaseManager()
load_dotenv()

True

### **Create HydroShare Datasets: Download HydroShare Repositories**

In [ ]:
def authenticate_hydroshare():
    hs_user = os.getenv('HYDROSHARE_USERNAME')
    hs_pass = os.getenv('HYDROSHARE_PASSWORD')

    if not hs_user or not hs_pass:
        raise EnvironmentError("System Error: HydroShare credentials not found in environment variables.")

    # Bypasses the interactive prompt by passing credentials directly
    hs = HydroShare(username=hs_user, password=hs_pass)
    
    return hs

In [ ]:
hs_session = authenticate_hydroshare()
print("Authentication layer verified.")

In [ ]:
def download_raw_corpus(hs_client, resource_id: str, base_path: str = "./HydroShare-CIROH-Datasets") -> bool:
    """
    Downloads the 'Level Zero' artifacts for a HydroShare resource to local disk.
    """
    res_dir = os.path.join(base_path, resource_id)
    os.makedirs(res_dir, exist_ok=True)
    
    try:
        res = hs_client.resource(resource_id)
        
        # 1. Download Native Pydantic Dump (meta)
        try:
            meta_dict = res.metadata.model_dump() if hasattr(res.metadata, 'model_dump') else res.metadata.dict()
            with open(os.path.join(res_dir, "meta_dump.json"), 'w', encoding='utf-8') as f:
                json.dump(meta_dict, f, default=str, indent=4)
        except Exception as e:
            print(f"  -> Warning: Failed to dump meta for {resource_id}: {e}")

        # 2. Download Native System Metadata (sys_meta)
        try:
            with open(os.path.join(res_dir, "sys_meta_dump.json"), 'w', encoding='utf-8') as f:
                json.dump(res.system_metadata(), f, indent=4)
        except Exception as e:
            print(f"  -> Warning: Failed to dump sys_meta for {resource_id}: {e}")

        # 3. Download Raw XML/RDF 
        try:
            raw_xml = hs_client._hs_session.retrieve_string(res.metadata_path)
            with open(os.path.join(res_dir, "raw_metadata.xml"), 'w', encoding='utf-8') as f:
                f.write(raw_xml)
        except Exception as e:
            print(f"  -> Warning: Failed to download raw XML for {resource_id}: {e}")

        # 4. Download Physical README and capture file inventory
        file_inventory = []
        try:
            for f in res.files():
                file_inventory.append({
                    "file_name": str(f.name),
                    "file_path": str(f.path),
                    "extension": str(f.extension),
                    "checksum": str(f.checksum)
                })
                
                if str(f.name).lower().startswith("readme") and str(f.extension).lower() in ['.md', '.txt']:
                    try:
                        content = hs_client._hs_session.retrieve_string(f.url)
                        with open(os.path.join(res_dir, f.name), 'w', encoding='utf-8') as readme_f:
                            readme_f.write(content)
                    except Exception as e:
                        print(f"  -> Warning: Failed to download {f.name}: {e}")
                        
        except IndexError:
            # Crucial control structure for CollectionResources and ToolResources
            print(f"  -> Warning: No physical files found (IndexError mitigated).")
        except Exception as file_e:
             print(f"  -> Warning: File inventory failed: {file_e}")

        with open(os.path.join(res_dir, "file_inventory.json"), 'w', encoding='utf-8') as f:
             json.dump(file_inventory, f, indent=4)

        return True

    except Exception as e:
        print(f"Critical Failure downloading {resource_id}: {str(e)}")
        return False

In [ ]:
# Define file paths
csv_file_path = 'HydroShare_IDs_03272026.csv'
base_corpus_path = './HydroShare-CIROH-Datasets'

# 1. Read and validate IDs from the CSV file
ciroh_corpus_ids = []

try:
    with open(csv_file_path, 'r', encoding='utf-8') as file:
        for line in file:
            # Strip whitespace and newline characters
            clean_id = line.strip()
            
            # Validate standard HydroShare ID length (32 characters)
            if len(clean_id) == 32:
                ciroh_corpus_ids.append(clean_id)
            elif clean_id:
                print(f"Warning: Anomalous ID format skipped -> {clean_id}")
                
except FileNotFoundError:
    print(f"Error: File '{csv_file_path}' not found. Please check the directory path.")

print(f"Total valid IDs to process: {len(ciroh_corpus_ids)}\n")
print("-" * 50)

# 2. Orchestrate the Level Zero baseline download
success_count = 0
failed_ids = []

# Note: hs_session must be an authenticated HydroShare instance 
for index, res_id in enumerate(ciroh_corpus_ids, start=1):
    print(f"[{index}/{len(ciroh_corpus_ids)}] Processing resource: {res_id}")
    
    # Execute the extraction function defined previously
    success = download_raw_corpus(hs_session, res_id, base_path=base_corpus_path)
    
    if success:
        success_count += 1
    else:
        failed_ids.append(res_id)

# 3. Execution reporting
print("\n" + "=" * 50)
print("CORPUS ACQUISITION REPORT")
print("=" * 50)
print(f"Total processed: {len(ciroh_corpus_ids)}")
print(f"Successful downloads: {success_count}")
print(f"Failed downloads: {len(failed_ids)}")

if failed_ids:
    print("\nList of failed IDs (check network connection or access permissions):")
    for fid in failed_ids:
        print(f" - {fid}")

### **Create HydroShare Datasets: Artifacts**

In [50]:
# =========================
# Configuration
# =========================

BASE_DATASETS_DIR = Path("./HydroShare-CIROH-Datasets")
OUTPUT_PATH = Path("json/dataset_artifacts.json")

DATASET_ARTIFACT_TYPE_ID = 3  # TBLArtifactTypes -> HydroShare Dataset
HS_ID_RE = re.compile(r"^[0-9a-fA-F]{32}$")

In [51]:
# =========================
# Helpers
# =========================

def clean_text(value: Any) -> Any:
    if value is None:
        return None
    if not isinstance(value, str):
        return value
    value = value.replace("\x00", "")
    value = value.strip()
    return value if value else None


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_list(value: Any) -> List[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return []


def infer_document_type(resource_type: Optional[str]) -> str:
    if resource_type == "CollectionResource":
        return "HydroShare dataset collection"
    if resource_type == "ToolResource":
        return "HydroShare web app connector"
    return "HydroShare dataset resource"


def build_dataset_artifact_record(resource_dir: Path, id_artifact: int) -> Dict[str, Any]:
    resource_id = resource_dir.name

    meta = read_json(resource_dir / "meta_dump.json", {})
    sys_meta = read_json(resource_dir / "sys_meta_dump.json", {})

    resource_type = clean_text(meta.get("type")) or "CompositeResource"

    title = clean_text(meta.get("title")) or f"HydroShare Resource {resource_id}"
    url = (
        clean_text(meta.get("url"))
        or clean_text(sys_meta.get("resource_url"))
        or f"http://www.hydroshare.org/resource/{resource_id}"
    )

    abstract = clean_text(meta.get("abstract")) or ""
    keywords = normalize_list(meta.get("subjects"))
    if not isinstance(keywords, list):
        keywords = []

    record = {
        "idArtifact": id_artifact,
        "idArtifactType": DATASET_ARTIFACT_TYPE_ID,
        "Title": title,
        "URL": url,
        "idArtifactParent": None,
        "abstract": abstract,
        "keywords": keywords,
        "document_type": infer_document_type(resource_type),
        "resource_id": resource_id,
        "resource_type": resource_type,
        "identifier": clean_text(meta.get("identifier")),
        "language": clean_text(meta.get("language")),
        "doi": clean_text(sys_meta.get("doi")),
        "date_created": clean_text(sys_meta.get("date_created")),
        "date_last_updated": clean_text(sys_meta.get("date_last_updated")),
        "published": bool(sys_meta.get("published", False)),
        "immutable": bool(sys_meta.get("immutable", False)),
        "public": bool(sys_meta.get("public", False)),
        "discoverable": bool(sys_meta.get("discoverable", False)),
        "shareable": bool(sys_meta.get("shareable", False)),
    }

    return record

In [52]:
# =========================
# dataset_artifacts.json
# =========================

if not BASE_DATASETS_DIR.exists():
    raise FileNotFoundError(f"Base directory not found: {BASE_DATASETS_DIR}")

resource_dirs = sorted(
    [
        p for p in BASE_DATASETS_DIR.iterdir()
        if p.is_dir() and HS_ID_RE.match(p.name)
    ],
    key=lambda p: p.name.lower()
)

print(f"Resources detected: {len(resource_dirs)}")

dataset_artifacts = []
errors = []

for idx, resource_dir in enumerate(resource_dirs, start=1):
    try:
        rec = build_dataset_artifact_record(resource_dir, id_artifact=idx)
        dataset_artifacts.append(rec)
    except Exception as e:
        errors.append({
            "resource_id": resource_dir.name,
            "error": str(e)
        })

print(f"Artifacts built: {len(dataset_artifacts)}")
print(f"Errors: {len(errors)}")

if errors:
    print("Error samples:")
    for e in errors[:10]:
        print(e)

Resources detected: 42
Artifacts built: 42
Errors: 0


In [53]:
# =========================
# Persistence
# =========================

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset_artifacts, f, indent=2, ensure_ascii=False)

print(f"File saved to: {OUTPUT_PATH}")
print(f"Total records: {len(dataset_artifacts)}")

File saved to: json/dataset_artifacts.json
Total records: 42


In [54]:
# =========================
# Quick inspection
# =========================

if dataset_artifacts:
    print(json.dumps(dataset_artifacts[0], indent=2, ensure_ascii=False)[:4000])

{
  "idArtifact": 1,
  "idArtifactType": 3,
  "Title": "Enabling collaboration through data and model sharing with CUAHSI HydroShare; CIROH Devcon 25 presentation",
  "URL": "http://www.hydroshare.org/resource/023bfa101586432ba0f6ed9ddfea60a9",
  "idArtifactParent": null,
  "abstract": "Collaboration is central to CIROH. Advancing hydrology research to operations relies on model and data sharing, requiring open data integration, accessible computing, and teamwork. The CUAHSI HydroShare platform was developed to enable researchers to share digital products from their research, including data, models, and workflows, in line with Findable, Accessible, Interoperable, and Reusable (FAIR) principles. This project advances HydroShare for CIROH collaborative research and education with objectives to (1) enhance community data access; (2) establish interoperability with scalable computing; (3) demonstrate computational reproducibility; and (4) establish and grow a CIROH Community on HydroShare.

### **Create HydroShare Datasets: Chunks**

In [88]:
# =========================
# Configuration
# =========================

BASE_DATASETS_DIR = Path("./HydroShare-CIROH-Datasets")
ARTIFACTS_PATH = Path("json/dataset_artifacts.json")
OUTPUT_CHUNKS_PATH = Path("json/dataset_chunks.json")

DATASET_ARTIFACT_TYPE_ID = 3

HS_ID_RE = re.compile(r"^[0-9a-fA-F]{32}$")
HS_RESOURCE_ID_RE = re.compile(r"/resource/([0-9a-fA-F]{32})")
URL_RE = re.compile(r"https?://[^\s<>\"]+")

NS = {
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "hsterms": "https://www.hydroshare.org/terms/",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
}

In [89]:
# =========================
# Resolve dataset chunk types from DB
# =========================

with db_manager as db:
    rows = db.execute_query(
        """
        SELECT idchunktype, typename
        FROM tblchunktypes
        WHERE idartifacttype = %s;
        """,
        (DATASET_ARTIFACT_TYPE_ID,),
        fetch=True
    ) or []

if not rows:
    raise RuntimeError(
        f"No chunk types found in TBLChunkTypes for idArtifactType={DATASET_ARTIFACT_TYPE_ID}"
    )

chunk_type_name_to_id: Dict[str, int] = {
    r["typename"]: r["idchunktype"] for r in rows
}

required_chunk_names = {
    "Abstract",
    "Spatial Coverage",
    "Temporal Coverage",
    "Variable Metadata",
    "Data Services Info",
    "File Description",
    "Collection Contents",
    "Subject Keywords",
    "Related Resources Context",
    "Citation",
    "Creator",
    "Contributor",
    "Funding Agency",
    "Additional Metadata",
    "Readme File",
}

missing_names = required_chunk_names - set(chunk_type_name_to_id.keys())
if missing_names:
    raise RuntimeError(
        "Missing required dataset chunk types in DB: " + ", ".join(sorted(missing_names))
    )

print("Loaded dataset chunk types from DB:")
for name, chunk_id in sorted(chunk_type_name_to_id.items(), key=lambda x: x[1]):
    print(f" - {name}: {chunk_id}")

Loaded dataset chunk types from DB:
 - Abstract: 12
 - Spatial Coverage: 13
 - Temporal Coverage: 14
 - Variable Metadata: 15
 - Data Services Info: 16
 - File Description: 17
 - Collection Contents: 18
 - Subject Keywords: 19
 - Related Resources Context: 20
 - Citation: 42
 - Creator: 43
 - Contributor: 44
 - Funding Agency: 45
 - Additional Metadata: 46
 - Readme File: 47


In [90]:
# =========================
# Load dataset_artifacts.json and build resource_id -> idArtifact map
# =========================

with open(ARTIFACTS_PATH, "r", encoding="utf-8") as f:
    dataset_artifacts = json.load(f)

resource_to_artifact_id: Dict[str, int] = {}
for rec in dataset_artifacts:
    resource_id = rec.get("resource_id")
    id_artifact = rec.get("idArtifact")
    if resource_id and id_artifact is not None:
        resource_to_artifact_id[str(resource_id)] = int(id_artifact)

print(f"Loaded dataset artifacts: {len(dataset_artifacts)}")
print(f"Mapped resource_id -> idArtifact: {len(resource_to_artifact_id)}")

Loaded dataset artifacts: 42
Mapped resource_id -> idArtifact: 42


In [91]:
# =========================
# Generic helpers
# =========================

def clean_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    if not isinstance(value, str):
        value = str(value)
    value = value.replace("\x00", "")
    value = html.unescape(value)
    value = value.strip()
    return value or None


def clean_multiline_text(value: Any) -> str:
    value = clean_text(value) or ""
    value = re.sub(r"\n{3,}", "\n\n", value)
    return value


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_list(value: Any) -> List[Any]:
    if isinstance(value, list):
        return value
    return []


def normalize_url_token(url: str) -> str:
    return url.rstrip(".,;:)]}>\"'")


def extract_urls(text: Optional[str]) -> List[str]:
    if not text:
        return []
    urls = []
    seen = set()
    for raw_url in URL_RE.findall(text):
        clean_url = normalize_url_token(raw_url)
        if clean_url and clean_url not in seen:
            seen.add(clean_url)
            urls.append(clean_url)
    return urls


def extract_hydroshare_resource_id(text: Optional[str]) -> Optional[str]:
    if not text:
        return None
    m = HS_RESOURCE_ID_RE.search(text)
    return m.group(1).lower() if m else None


def stringify_list(values: List[Any]) -> List[str]:
    out = []
    for v in values:
        if v is None:
            continue
        s = clean_text(v)
        if s:
            out.append(s)
    return out


def limited_join(items: List[str], max_items: int = 12) -> str:
    items = [x for x in items if x]
    if not items:
        return ""
    if len(items) <= max_items:
        return ", ".join(items)
    head = ", ".join(items[:max_items])
    remaining = len(items) - max_items
    return f"{head}, and {remaining} more"


def file_category(extension: str) -> str:
    ext = (extension or "").lower()

    if ext in {".py", ".ipynb", ".r", ".m", ".jl", ".java", ".cpp", ".c", ".sh"}:
        return "code_and_notebooks"
    if ext in {".tif", ".tiff", ".vrt", ".shp", ".gpkg", ".geojson", ".nc"}:
        return "geospatial_data"
    if ext in {".csv", ".tsv", ".xlsx", ".xls", ".parquet"}:
        return "tabular_data"
    if ext in {".pdf", ".md", ".txt", ".docx", ".pptx", ".ppt"}:
        return "documents"
    if ext in {".png", ".jpg", ".jpeg", ".gif", ".svg"}:
        return "images_media"
    if ext in {".zip", ".tar", ".gz", ".bz2", ".7z"}:
        return "archives"
    return "other"


def normalize_relation_label(label: Optional[str]) -> Optional[str]:
    if not label:
        return None

    label = clean_text(label)
    if not label:
        return None

    mapping = {
        "The content of this resource references": "references",
        "This resource is referenced by": "isReferencedBy",
        "The content of this resource can be executed by": "isExecutedBy",
        "This resource includes": "hasPart",
        "This collection is described by": "isDescribedBy",
        "The content of this resource is derived from": "source",
        "This resource is part of": "isPartOf",
    }
    return mapping.get(label, label)


def relation_display_label(label: Optional[str]) -> str:
    label = clean_text(label) or "related"
    mapping = {
        "references": "This resource references",
        "isReferencedBy": "This resource is referenced by",
        "isExecutedBy": "This resource can be executed by",
        "hasPart": "This resource includes",
        "isDescribedBy": "This resource is described by",
        "source": "This resource is derived from",
        "isPartOf": "This resource is part of",
    }
    return mapping.get(label, label)

In [92]:
# =========================
# XML helpers
# =========================

def _split_tag(tag: str) -> Tuple[Optional[str], str]:
    if tag.startswith("{"):
        uri, local = tag[1:].split("}", 1)
        return uri, local
    return None, tag


def _qname(tag: str) -> str:
    uri, local = _split_tag(tag)
    if uri is None:
        return local
    for prefix, ns_uri in NS.items():
        if uri == ns_uri:
            return f"{prefix}:{local}"
    return local


def _elem_value(elem: Optional[ET.Element]) -> Tuple[Optional[str], Optional[str]]:
    if elem is None:
        return None, None

    resource_attr = elem.attrib.get(f"{{{NS['rdf']}}}resource")
    if resource_attr:
        return clean_text(resource_attr), "uri_ref"

    text = clean_text("".join(elem.itertext()))
    if text:
        return text, "literal"

    return None, None


def parse_xml_for_chunks(xml_path: Path) -> Dict[str, Any]:
    out = {
        "resource_xml_type": None,
        "typed_relations": [],
        "geospatial_relations": [],
        "collection_members": [],
        "described_by": [],
        "tool_config": None,
    }

    if not xml_path.exists():
        return out

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception:
        return out

    resource_elem = None
    for child in root:
        qn = _qname(child.tag)
        if qn in {"hsterms:CompositeResource", "hsterms:CollectionResource", "hsterms:ToolResource"}:
            resource_elem = child
            out["resource_xml_type"] = qn.replace("hsterms:", "")
            break

    if resource_elem is None:
        return out

    typed_relations = []
    for relation_container in resource_elem.findall(f"{{{NS['dc']}}}relation"):
        desc = next(iter(relation_container), None)
        if desc is None:
            continue

        for rel_elem in desc:
            target, _ = _elem_value(rel_elem)
            if not target:
                continue
            typed_relations.append({
                "predicate_qname": _qname(rel_elem.tag),
                "normalized_label": normalize_relation_label(_qname(rel_elem.tag).split(":")[-1]),
                "target": target,
                "target_resource_id": extract_hydroshare_resource_id(target),
            })

    seen_rel = set()
    dedup_rel = []
    for rel in typed_relations:
        key = (rel["normalized_label"], rel["target"])
        if key not in seen_rel:
            seen_rel.add(key)
            dedup_rel.append(rel)
    out["typed_relations"] = dedup_rel

    geospatial_relations = []
    for geo_container in resource_elem.findall(f"{{{NS['hsterms']}}}geospatialRelation"):
        desc = next(iter(geo_container), None)
        if desc is None:
            continue

        relation_name = None
        target = None
        for child in desc:
            child_qname = _qname(child.tag)
            if child_qname == "hsterms:relation_name":
                relation_name = clean_text("".join(child.itertext()))
            else:
                value, _ = _elem_value(child)
                if value:
                    target = value

        if target:
            geospatial_relations.append({
                "name": relation_name,
                "target": target,
            })

    out["geospatial_relations"] = geospatial_relations

    collection_members = []
    described_by = []
    for rel in out["typed_relations"]:
        if rel["predicate_qname"] == "dcterms:hasPart":
            collection_members.append({
                "target": rel["target"],
                "target_resource_id": rel["target_resource_id"],
            })
        elif rel["predicate_qname"] == "hsterms:isDescribedBy":
            described_by.append(rel["target"])

    out["collection_members"] = collection_members
    out["described_by"] = described_by

    def find_hsterms_value(tag_name: str) -> Optional[str]:
        parent = resource_elem.find(f"{{{NS['hsterms']}}}{tag_name}")
        if parent is None:
            return None
        value_elem = parent.find(f".//{{{NS['hsterms']}}}value")
        value, _ = _elem_value(value_elem)
        return value

    if out["resource_xml_type"] == "ToolResource":
        out["tool_config"] = {
            "app_home_page_url": find_hsterms_value("AppHomePageUrl"),
            "request_url_base": find_hsterms_value("RequestUrlBase"),
            "request_url_base_file": find_hsterms_value("RequestUrlBaseFile"),
            "tool_version": find_hsterms_value("ToolVersion"),
            "testing_protocol_url": find_hsterms_value("TestingProtocolUrl"),
        }

    return out

In [94]:
# =========================
# README helpers
# =========================

README_ALLOWED_SUFFIXES = {"readme.md", "readme.txt"}
README_HEADING_RE = re.compile(r"^(#{1,6})\s+(.*\S)\s*$")
README_RULE_RE = re.compile(r"^\s*([-*_])\1{2,}\s*$")


def normalize_readme_text_for_dedup(text: str) -> str:
    """
    Normalize README text so equivalent files can be deduplicated by content.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    text = text.strip()
    return text


def get_readme_files(resource_dir: Path) -> List[Tuple[str, str]]:
    """
    Returns a list of unique (actual_filename_on_disk, text) tuples for README files.
    Deduplicates:
      1) by real filesystem path
      2) by normalized content
    This avoids duplicate processing on case-insensitive filesystems.
    """
    results: List[Tuple[str, str]] = []
    seen_real_paths = set()
    seen_contents = set()

    # Look only at real files that exist in the directory
    for path in sorted(resource_dir.iterdir(), key=lambda p: p.name.lower()):
        if not path.is_file():
            continue

        if path.name.lower() not in README_ALLOWED_SUFFIXES:
            continue

        real_path = str(path.resolve())
        if real_path in seen_real_paths:
            continue
        seen_real_paths.add(real_path)

        text = path.read_text(encoding="utf-8", errors="replace").replace("\x00", "")
        normalized_text = normalize_readme_text_for_dedup(text)

        if not normalized_text:
            continue

        if normalized_text in seen_contents:
            continue
        seen_contents.add(normalized_text)

        results.append((path.name, text))

    return results

def parse_readme_sections(readme_text: str, source_file: str) -> List[Dict[str, Any]]:
    """
    Parse README content into hierarchical sections using markdown headings.

    Rules:
    - Preserve the real heading depth (#, ##, ###, ####, #####, ######)
    - Text before the first heading becomes a root chunk titled 'README Introduction'
    - Markdown separator lines like --- are ignored when they appear alone
    - Headings with no body are still emitted if they have children
    """
    text = readme_text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")

    specs: List[Dict[str, Any]] = []
    next_local_id = 1

    preamble_lines: List[str] = []
    current_section: Optional[Dict[str, Any]] = None
    stack: List[Tuple[int, int]] = []  # (heading_level, temp_local_id)

    def clean_section_body(section_lines: List[str]) -> str:
        cleaned = []
        for ln in section_lines:
            if README_RULE_RE.match(ln):
                continue
            cleaned.append(ln)
        return "\n".join(cleaned).strip()

    def flush_current_section():
        nonlocal current_section
        if current_section is None:
            return

        body = clean_section_body(current_section["lines"])

        # Emit the section if it has body OR if it has children
        if body or current_section.get("has_children", False):
            if not body:
                # keep a non-empty chunk_text so later embedding code does not fail
                body = current_section["title"]

            specs.append({
                "temp_local_id": current_section["temp_local_id"],
                "temp_parent_local_id": current_section["temp_parent_local_id"],
                "chunk_type_name": "Readme File",
                "chunk_text": body,
                "section_hint": current_section["title"],
                "source_files": [source_file],
                "source_fields": ["readme_text"],
                "extra_type_specific": {
                    "source_file": source_file,
                    "heading_level": current_section["heading_level"],
                    "heading_marker": current_section["heading_marker"],
                    "readme_chunk_kind": "markdown_section",
                    "has_children": bool(current_section.get("has_children", False)),
                    "has_body": bool(clean_section_body(current_section["lines"])),
                }
            })

        current_section = None

    def create_intro_chunk(intro_lines: List[str]):
        nonlocal next_local_id
        intro_text = clean_section_body(intro_lines)
        if not intro_text:
            return

        specs.append({
            "temp_local_id": next_local_id,
            "temp_parent_local_id": None,
            "chunk_type_name": "Readme File",
            "chunk_text": intro_text,
            "section_hint": "README Introduction",
            "source_files": [source_file],
            "source_fields": ["readme_text"],
            "extra_type_specific": {
                "source_file": source_file,
                "heading_level": 0,
                "heading_marker": None,
                "readme_chunk_kind": "readme_introduction",
                "has_children": False,
                "has_body": True,
            }
        })
        next_local_id += 1

    for line in lines:
        heading_match = README_HEADING_RE.match(line)

        if heading_match:
            hashes = heading_match.group(1)
            title = heading_match.group(2).strip()
            heading_level = len(hashes)
            heading_marker = hashes

            if current_section is None and preamble_lines:
                create_intro_chunk(preamble_lines)
                preamble_lines = []

            # If the new heading is deeper, the current section becomes a parent
            if current_section is not None and heading_level > current_section["heading_level"]:
                current_section["has_children"] = True

            flush_current_section()

            while stack and stack[-1][0] >= heading_level:
                stack.pop()

            parent_local_id = stack[-1][1] if stack else None

            current_section = {
                "temp_local_id": next_local_id,
                "temp_parent_local_id": parent_local_id,
                "title": title,
                "heading_level": heading_level,
                "heading_marker": heading_marker,
                "lines": [],
                "has_children": False,
            }

            stack.append((heading_level, next_local_id))
            next_local_id += 1

        else:
            if current_section is None:
                preamble_lines.append(line)
            else:
                current_section["lines"].append(line)

    if current_section is None and preamble_lines:
        create_intro_chunk(preamble_lines)
    else:
        flush_current_section()

    return specs

def build_readme_file_chunks(resource_dir: Path) -> List[Dict[str, Any]]:
    """
    Build README-derived chunk specs for one resource.
    Returns specs with temporary local ids so parent-child relationships can be resolved later.
    """
    all_specs: List[Dict[str, Any]] = []

    readme_files = get_readme_files(resource_dir)
    if not readme_files:
        return all_specs

    for source_file, readme_text in readme_files:
        text = readme_text.strip()
        if not text:
            continue

        specs = parse_readme_sections(text, source_file=source_file)
        all_specs.extend(specs)

    return all_specs

In [95]:
# =========================
# Chunk builders
# =========================

def build_abstract_chunk(meta: dict) -> Optional[Dict[str, Any]]:
    abstract = clean_multiline_text(meta.get("abstract"))
    if not abstract:
        return None

    return {
        "chunk_type_name": "Abstract",
        "chunk_text": abstract,
        "section_hint": "HydroShare abstract",
        "source_files": ["meta_dump.json"],
        "source_fields": ["abstract"],
    }


def build_spatial_chunks(meta: dict, xml_info: dict) -> List[Dict[str, Any]]:
    chunks = []

    spatial = meta.get("spatial_coverage")
    if isinstance(spatial, dict):
        cov_type = clean_text(spatial.get("type"))
        north = spatial.get("northlimit") or spatial.get("north")
        south = spatial.get("southlimit") or spatial.get("south")
        east = spatial.get("eastlimit") or spatial.get("east")
        west = spatial.get("westlimit") or spatial.get("west")
        units = clean_text(spatial.get("units"))
        projection = clean_text(spatial.get("projection"))

        parts = []
        if cov_type:
            parts.append(f"This resource has {cov_type}-based spatial coverage.")
        if any(v is not None for v in [north, south, east, west]):
            parts.append(
                f"Its bounding coordinates are north {north}, south {south}, east {east}, and west {west}."
            )
        if units:
            parts.append(f"The coordinate units are {units}.")
        if projection:
            parts.append(f"The reference system is {projection}.")

        text = " ".join(parts).strip()
        if text:
            chunks.append({
                "chunk_type_name": "Spatial Coverage",
                "chunk_text": text,
                "section_hint": "Spatial coverage",
                "source_files": ["meta_dump.json"],
                "source_fields": ["spatial_coverage"],
            })

    for rel in xml_info.get("geospatial_relations") or []:
        name = clean_text(rel.get("name"))
        target = clean_text(rel.get("target"))
        if name and target:
            text = f"This resource has a geospatial relation to {name} via {target}."
        elif target:
            text = f"This resource has a geospatial relation via {target}."
        else:
            continue

        chunks.append({
            "chunk_type_name": "Spatial Coverage",
            "chunk_text": text,
            "section_hint": "Geospatial relation",
            "source_files": ["raw_metadata.xml"],
            "source_fields": ["geospatial_relations"],
        })

    return chunks


def build_temporal_chunks(meta: dict) -> List[Dict[str, Any]]:
    chunks = []

    period = meta.get("period_coverage")
    if not isinstance(period, dict):
        return chunks

    start = clean_text(period.get("start"))
    end = clean_text(period.get("end"))
    if not start and not end:
        return chunks

    if start and end:
        text = f"The temporal coverage of this resource spans from {start} to {end}."
    elif start:
        text = f"The temporal coverage of this resource starts on {start}."
    else:
        text = f"The temporal coverage of this resource ends on {end}."

    chunks.append({
        "chunk_type_name": "Temporal Coverage",
        "chunk_text": text,
        "section_hint": "Temporal coverage",
        "source_files": ["meta_dump.json"],
        "source_fields": ["period_coverage"],
    })

    return chunks


def build_data_services_chunks(resource_id: str, sys_meta: dict, file_inventory: list, xml_info: dict) -> List[Dict[str, Any]]:
    chunks = []

    content_types = stringify_list(normalize_list(sys_meta.get("content_types")))
    if content_types:
        chunks.append({
            "chunk_type_name": "Data Services Info",
            "chunk_text": "This resource declares the following content types: " + limited_join(content_types) + ".",
            "section_hint": "Content types",
            "source_files": ["sys_meta_dump.json"],
            "source_fields": ["content_types"],
        })

    geo_exts = {".tif", ".tiff", ".vrt", ".shp", ".gpkg", ".geojson", ".nc"}
    has_geospatial_files = any(
        isinstance(f, dict) and str(f.get("extension", "")).lower() in geo_exts
        for f in file_inventory
    )
    if has_geospatial_files:
        wms = f"https://geoserver.hydroshare.org/geoserver/HS-{resource_id}/wms?request=GetCapabilities"
        wcs = f"https://geoserver.hydroshare.org/geoserver/HS-{resource_id}/wcs?request=GetCapabilities"

        chunks.append({
            "chunk_type_name": "Data Services Info",
            "chunk_text": f"This resource exposes a WMS service endpoint at {wms}.",
            "section_hint": "WMS service",
            "source_files": ["file_inventory.json"],
            "source_fields": ["file_inventory"],
        })
        chunks.append({
            "chunk_type_name": "Data Services Info",
            "chunk_text": f"This resource exposes a WCS service endpoint at {wcs}.",
            "section_hint": "WCS service",
            "source_files": ["file_inventory.json"],
            "source_fields": ["file_inventory"],
        })

    tool_config = xml_info.get("tool_config")
    if isinstance(tool_config, dict):
        mapping = [
            ("app_home_page_url", "App home page"),
            ("request_url_base", "Resource launch pattern"),
            ("request_url_base_file", "File launch pattern"),
            ("tool_version", "Tool version"),
            ("testing_protocol_url", "Testing protocol URL"),
        ]

        for field_name, label in mapping:
            value = clean_text(tool_config.get(field_name))
            if not value:
                continue

            chunks.append({
                "chunk_type_name": "Data Services Info",
                "chunk_text": f"{label}: {value}.",
                "section_hint": label,
                "source_files": ["raw_metadata.xml"],
                "source_fields": ["tool_config"],
            })

    return chunks


def build_file_description_chunks(file_inventory: list) -> List[Dict[str, Any]]:
    chunks = []

    if not isinstance(file_inventory, list):
        return chunks

    for f in file_inventory:
        if not isinstance(f, dict):
            continue

        file_name = clean_text(f.get("file_name"))
        file_path = clean_text(f.get("file_path"))
        extension = clean_text(f.get("extension"))
        checksum = clean_text(f.get("checksum"))

        if not file_name and not file_path:
            continue

        parts = []
        if file_name:
            parts.append(f"File name: {file_name}.")
        if file_path:
            parts.append(f"Path: {file_path}.")
        if extension:
            parts.append(f"Extension: {str(extension).lower()}.")
        if checksum:
            parts.append(f"Checksum: {checksum}.")

        text = " ".join(parts).strip()
        if not text:
            continue

        chunks.append({
            "chunk_type_name": "File Description",
            "chunk_text": text,
            "section_hint": file_name or "Repository file",
            "source_files": ["file_inventory.json"],
            "source_fields": ["file_inventory"],
            "extra_type_specific": {
                "file_name": file_name,
                "file_path": file_path,
                "extension": str(extension).lower() if extension else None,
                "checksum": checksum,
                "file_category": file_category(extension or ""),
            }
        })

    return chunks


def build_collection_contents_chunks(xml_info: dict) -> List[Dict[str, Any]]:
    chunks = []

    members = xml_info.get("collection_members") or []
    for member in members:
        target = clean_text(member.get("target"))
        target_resource_id = clean_text(member.get("target_resource_id"))
        if not target:
            continue

        text = f"This collection includes the member resource: {target}."
        chunks.append({
            "chunk_type_name": "Collection Contents",
            "chunk_text": text,
            "section_hint": "Collection member",
            "source_files": ["raw_metadata.xml"],
            "source_fields": ["collection_members"],
            "extra_type_specific": {
                "target_resource_id": target_resource_id,
            }
        })

    for desc in xml_info.get("described_by") or []:
        desc = clean_text(desc)
        if not desc:
            continue

        chunks.append({
            "chunk_type_name": "Collection Contents",
            "chunk_text": f"This collection is described by: {desc}.",
            "section_hint": "Collection description",
            "source_files": ["raw_metadata.xml"],
            "source_fields": ["described_by"],
        })

    return chunks


def build_subject_keywords_chunk(meta: dict) -> Optional[Dict[str, Any]]:
    subjects = stringify_list(normalize_list(meta.get("subjects")))
    if not subjects:
        return None

    return {
        "chunk_type_name": "Subject Keywords",
        "chunk_text": "Subject keywords associated with this resource include " + limited_join(subjects, max_items=20) + ".",
        "section_hint": "Subject keywords",
        "source_files": ["meta_dump.json"],
        "source_fields": ["subjects"],
    }


def build_citation_chunks(meta: dict) -> List[Dict[str, Any]]:
    chunks = []

    citation = clean_text(meta.get("citation"))
    if citation:
        chunks.append({
            "chunk_type_name": "Citation",
            "chunk_text": f"Recommended citation: {citation}.",
            "section_hint": "Citation",
            "source_files": ["meta_dump.json"],
            "source_fields": ["citation"],
        })

    rights = meta.get("rights")
    if isinstance(rights, dict):
        statement = clean_text(rights.get("statement"))
        url = clean_text(rights.get("url"))

        parts = []
        if statement:
            parts.append(f"License statement: {statement.rstrip('.')}.")
        if url:
            parts.append(f"License URL: {url.rstrip('.')}.")
        text = " ".join(parts).strip()

        if text:
            chunks.append({
                "chunk_type_name": "Citation",
                "chunk_text": text,
                "section_hint": "License",
                "source_files": ["meta_dump.json"],
                "source_fields": ["rights"],
            })

    return chunks


def build_creator_chunks(meta: dict) -> List[Dict[str, Any]]:
    chunks = []

    creators = normalize_list(meta.get("creators"))
    for creator in creators:
        if not isinstance(creator, dict):
            continue

        name = clean_text(creator.get("name"))
        if not name:
            continue

        parts = [f"Creator: {name}."]
        organization = clean_text(creator.get("organization"))
        email = clean_text(creator.get("email"))
        homepage = clean_text(creator.get("homepage"))
        address = clean_text(creator.get("address"))
        phone = clean_text(creator.get("phone"))
        creator_order = creator.get("creator_order")

        if creator_order is not None:
            parts.append(f"Creator order: {creator_order}.")
        if organization:
            parts.append(f"Organization: {organization}.")
        if email:
            parts.append(f"Email: {email}.")
        if homepage:
            parts.append(f"Homepage: {homepage}.")
        if address:
            parts.append(f"Address: {address}.")
        if phone:
            parts.append(f"Phone: {phone}.")

        identifiers = creator.get("identifiers") if isinstance(creator.get("identifiers"), dict) else {}
        for id_type, id_value in identifiers.items():
            id_type_clean = clean_text(id_type)
            id_value_clean = clean_text(id_value)
            if id_type_clean and id_value_clean:
                parts.append(f"{id_type_clean}: {id_value_clean}.")

        chunks.append({
            "chunk_type_name": "Creator",
            "chunk_text": " ".join(parts),
            "section_hint": name,
            "source_files": ["meta_dump.json"],
            "source_fields": ["creators"],
            "extra_type_specific": {
                "creator_name": name,
                "creator_order": creator_order,
                "hydroshare_user_id": creator.get("hydroshare_user_id"),
            }
        })

    return chunks


def build_contributor_chunks(meta: dict) -> List[Dict[str, Any]]:
    chunks = []

    contributors = normalize_list(meta.get("contributors"))
    for contributor in contributors:
        if not isinstance(contributor, dict):
            continue

        name = clean_text(contributor.get("name"))
        if not name:
            continue

        parts = [f"Contributor: {name}."]
        organization = clean_text(contributor.get("organization"))
        email = clean_text(contributor.get("email"))
        homepage = clean_text(contributor.get("homepage"))
        address = clean_text(contributor.get("address"))
        phone = clean_text(contributor.get("phone"))

        if organization:
            parts.append(f"Organization: {organization}.")
        if email:
            parts.append(f"Email: {email}.")
        if homepage:
            parts.append(f"Homepage: {homepage}.")
        if address:
            parts.append(f"Address: {address}.")
        if phone:
            parts.append(f"Phone: {phone}.")

        chunks.append({
            "chunk_type_name": "Contributor",
            "chunk_text": " ".join(parts),
            "section_hint": name,
            "source_files": ["meta_dump.json"],
            "source_fields": ["contributors"],
            "extra_type_specific": {
                "contributor_name": name,
            }
        })

    return chunks


def build_funding_chunks(meta: dict) -> List[Dict[str, Any]]:
    chunks = []

    awards = normalize_list(meta.get("awards"))
    for award in awards:
        if not isinstance(award, dict):
            continue

        agency = clean_text(award.get("funding_agency_name"))
        title = clean_text(award.get("title"))
        number = clean_text(award.get("number"))
        agency_url = clean_text(award.get("funding_agency_url"))

        parts = []
        if agency:
            parts.append(f"Funding agency: {agency}.")
        if title:
            parts.append(f"Award title: {title}.")
        if number:
            parts.append(f"Award number: {number}.")
        if agency_url:
            parts.append(f"Funding agency URL: {agency_url}.")

        text = " ".join(parts).strip()
        if not text:
            continue

        chunks.append({
            "chunk_type_name": "Funding Agency",
            "chunk_text": text,
            "section_hint": agency or "Funding agency",
            "source_files": ["meta_dump.json"],
            "source_fields": ["awards"],
            "extra_type_specific": {
                "funding_agency_name": agency,
                "funding_agency_url": agency_url,
                "award_number": number,
            }
        })

    return chunks


def build_additional_metadata_chunks(meta: dict) -> List[Dict[str, Any]]:
    chunks = []

    additional_metadata = meta.get("additional_metadata")
    if not isinstance(additional_metadata, dict):
        return chunks

    for key, value in additional_metadata.items():
        key_clean = clean_text(key)
        value_clean = clean_text(value)

        if not key_clean or not value_clean:
            continue

        text = f"{key_clean}: {value_clean.rstrip('.')}."

        chunks.append({
            "chunk_type_name": "Additional Metadata",
            "chunk_text": text,
            "section_hint": key_clean,
            "source_files": ["meta_dump.json"],
            "source_fields": ["additional_metadata"],
            "extra_type_specific": {
                "metadata_key": key_clean,
                "metadata_value": value_clean,
            }
        })

    return chunks


def build_related_resource_chunks(meta: dict, xml_info: dict) -> List[Dict[str, Any]]:
    chunks = []

    merged_relations = {}
    xml_only_relations = {}

    # Meta relations are preferred because they match what HydroShare displays
    for rel in normalize_list(meta.get("relations")):
        if not isinstance(rel, dict):
            continue

        raw_label = clean_text(rel.get("type"))
        target = clean_text(rel.get("value"))
        if not target:
            continue

        normalized_label = normalize_relation_label(raw_label)
        key = (normalized_label, target)

        merged_relations[key] = {
            "label_for_text": relation_display_label(normalized_label),
            "target": target,
            "target_resource_id": extract_hydroshare_resource_id(target),
            "embedded_urls": extract_urls(target),
            "source_files": ["meta_dump.json"],
            "source_fields": ["relations"],
        }

    # XML relations only if not already represented by meta_dump.json
    for rel in xml_info.get("typed_relations") or []:
        normalized_label = clean_text(rel.get("normalized_label"))
        target = clean_text(rel.get("target"))
        if not target:
            continue

        key = (normalized_label, target)
        if key in merged_relations:
            # relation already represented by meta_dump.json
            merged_relations[key]["source_files"] = ["meta_dump.json", "raw_metadata.xml"]
            merged_relations[key]["source_fields"] = ["relations", "typed_relations"]
            continue

        xml_only_relations[key] = {
            "label_for_text": relation_display_label(normalized_label),
            "target": target,
            "target_resource_id": clean_text(rel.get("target_resource_id")),
            "embedded_urls": extract_urls(target),
            "source_files": ["raw_metadata.xml"],
            "source_fields": ["typed_relations"],
        }

    all_relations = list(merged_relations.values()) + list(xml_only_relations.values())

    for rel in all_relations:
        chunks.append({
            "chunk_type_name": "Related Resources Context",
            "chunk_text": f"{rel['label_for_text']}: {rel['target']}.",
            "section_hint": rel["label_for_text"],
            "source_files": rel["source_files"],
            "source_fields": rel["source_fields"],
            "extra_type_specific": {
                "target_resource_id": rel["target_resource_id"],
                "embedded_urls": rel["embedded_urls"],
            }
        })

    return chunks


def build_chunk_record(
    *,
    id_artifact: int,
    id_chunk: int,
    order: int,
    chunk_type_name: str,
    chunk_text: str,
    section_hint: str,
    source_files: List[str],
    source_fields: List[str],
    id_chunk_parent: Optional[int] = None,
    supporting_quote: Optional[List[str]] = None,
    extra_type_specific: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    type_specific = {
        "source_files": source_files,
        "source_fields": source_fields,
    }
    if extra_type_specific:
        type_specific.update(extra_type_specific)

    return {
        "idArtifact": id_artifact,
        "idChunk": id_chunk,
        "order": order,
        "idChunkType": chunk_type_name_to_id[chunk_type_name],
        "chunk_text": chunk_text,
        "idChunkParent": id_chunk_parent,
        "section_hint": section_hint,
        "supporting_quote": supporting_quote or [],
        "type_specific": type_specific,
    }

In [96]:
# =========================
# Build dataset_chunks.json
# =========================

if not BASE_DATASETS_DIR.exists():
    raise FileNotFoundError(f"Base folder not found: {BASE_DATASETS_DIR}")

resource_dirs = sorted(
    [
        p for p in BASE_DATASETS_DIR.iterdir()
        if p.is_dir() and HS_ID_RE.match(p.name)
    ],
    key=lambda p: p.name.lower()
)

dataset_chunks: List[Dict[str, Any]] = []
errors: List[Dict[str, Any]] = []

next_chunk_id = 1

for resource_dir in resource_dirs:
    resource_id = resource_dir.name
    id_artifact = resource_to_artifact_id.get(resource_id)

    if id_artifact is None:
        errors.append({
            "resource_id": resource_id,
            "error": "Missing idArtifact mapping from dataset_artifacts.json"
        })
        continue

    try:
        meta = read_json(resource_dir / "meta_dump.json", {})
        sys_meta = read_json(resource_dir / "sys_meta_dump.json", {})
        file_inventory = read_json(resource_dir / "file_inventory.json", [])
        xml_info = parse_xml_for_chunks(resource_dir / "raw_metadata.xml")

        order_counter = 1
        chunk_specs: List[Dict[str, Any]] = []

        abstract_chunk = build_abstract_chunk(meta)
        if abstract_chunk:
            chunk_specs.append(abstract_chunk)

        chunk_specs.extend(build_spatial_chunks(meta, xml_info))
        chunk_specs.extend(build_temporal_chunks(meta))

        # Variable Metadata intentionally disabled for now.

        chunk_specs.extend(build_data_services_chunks(resource_id, sys_meta, file_inventory, xml_info))
        chunk_specs.extend(build_file_description_chunks(file_inventory))
        chunk_specs.extend(build_collection_contents_chunks(xml_info))

        subjects_chunk = build_subject_keywords_chunk(meta)
        if subjects_chunk:
            chunk_specs.append(subjects_chunk)

        chunk_specs.extend(build_citation_chunks(meta)) 
        chunk_specs.extend(build_creator_chunks(meta))
        chunk_specs.extend(build_contributor_chunks(meta))
        chunk_specs.extend(build_funding_chunks(meta))
        chunk_specs.extend(build_additional_metadata_chunks(meta))
        chunk_specs.extend(build_related_resource_chunks(meta, xml_info))

        # README-derived chunks
        readme_specs = build_readme_file_chunks(resource_dir)
        chunk_specs.extend(readme_specs)

        # Resolve real idChunk values first, so README parent-child links can be mapped correctly
        temp_local_to_real_chunk_id: Dict[int, int] = {}
        projected_chunk_id = next_chunk_id

        for spec in chunk_specs:
            if "temp_local_id" in spec:
                temp_local_to_real_chunk_id[spec["temp_local_id"]] = projected_chunk_id
            projected_chunk_id += 1

        for spec in chunk_specs:
            parent_real_id = None
            temp_parent_local_id = spec.get("temp_parent_local_id")
            if temp_parent_local_id is not None:
                parent_real_id = temp_local_to_real_chunk_id.get(temp_parent_local_id)

            dataset_chunks.append(
                build_chunk_record(
                    id_artifact=id_artifact,
                    id_chunk=next_chunk_id,
                    order=order_counter,
                    chunk_type_name=spec["chunk_type_name"],
                    chunk_text=spec["chunk_text"],
                    section_hint=spec["section_hint"],
                    source_files=spec["source_files"],
                    source_fields=spec["source_fields"],
                    id_chunk_parent=parent_real_id,
                    supporting_quote=[],
                    extra_type_specific=spec.get("extra_type_specific"),
                )
            )
            next_chunk_id += 1
            order_counter += 1

    except Exception as e:
        errors.append({
            "resource_id": resource_id,
            "error": str(e)
        })

print(f"Chunks built: {len(dataset_chunks)}")
print(f"Resources processed: {len(resource_dirs)}")
print(f"Errors: {len(errors)}")

if errors:
    print("Sample errors:")
    for e in errors[:10]:
        print(e)

Chunks built: 1478
Resources processed: 42
Errors: 0


In [97]:
# =========================
# Save dataset_chunks.json
# =========================

OUTPUT_CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset_chunks, f, indent=2, ensure_ascii=False)

print(f"Saved: {OUTPUT_CHUNKS_PATH}")
print(f"Total chunk records: {len(dataset_chunks)}")

Saved: json/dataset_chunks.json
Total chunk records: 1478


In [98]:
# =========================
# Quick inspection
# =========================

if dataset_chunks:
    print(json.dumps(dataset_chunks[:20], indent=2, ensure_ascii=False)[:16000])

[
  {
    "idArtifact": 1,
    "idChunk": 1,
    "order": 1,
    "idChunkType": 12,
    "chunk_text": "Collaboration is central to CIROH. Advancing hydrology research to operations relies on model and data sharing, requiring open data integration, accessible computing, and teamwork. The CUAHSI HydroShare platform was developed to enable researchers to share digital products from their research, including data, models, and workflows, in line with Findable, Accessible, Interoperable, and Reusable (FAIR) principles. This project advances HydroShare for CIROH collaborative research and education with objectives to (1) enhance community data access; (2) establish interoperability with scalable computing; (3) demonstrate computational reproducibility; and (4) establish and grow a CIROH Community on HydroShare. This presentation will show the use of the CIROH 2i2c JupyterHub platform linked to HydroShare for accessing high community value datasets such as NOAA Analysis of Record for Calibrati